# Model comparison — per-subject × model fitting-criteria grid

For one **subject × model**, render a grid whose **columns are the fitting criteria** found on disk (pure MLE, joint MLE+Chi², Chi²-Noise, Chi²-Bound) and whose **rows are diagnostic plots** (loss distributions + the behavioral panels). The *interactive* cell picks one subject × model via dropdowns; the *batch* cell renders every combination and saves each to `results/RLModel/fig_model_cmp/`.

All logic lives in `model/compare.py` (+ `model/mle_reeval.py`); this notebook only loads data, sets config, and calls two entry points.

## Enable loading from relative packages

In [1]:
%load_ext autoreload
%autoreload 2
if "PKG" not in globals():
    root_parent_level = 2
    import importlib, sys, pathlib # https://stackoverflow.com/a/50395128/11996983
    PKG = %pwd
    PKG = pathlib.Path(PKG)
    root = PKG
    full_pkg = f"{root.name}"
    for _ in range(root_parent_level):
        root = root.parent
        full_pkg = f"{root.name}.{full_pkg}"
        MODULE_PATH = f"{root}{pathlib.os.path.sep}__init__.py"
        MODULE_NAME = f"{root.name}"
        spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
        module = importlib.util.module_from_spec(spec)
        sys.modules[spec.name] = module
        spec.loader.exec_module(module)
    __package__ = full_pkg

## Matplotlib backend + fonts

The comparison grids are large **static** images, so we use `%matplotlib inline` (not the `widget` backend the interactive model viewer uses) — no per-grid interactive canvas to manage, and the interactive cell's dropdowns re-render cleanly.

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]

## Load behavior + discover fits

`discover_fits()` globs `../../data/RLModel/`, parses each `mle_*`/`chisq_*` pickle into its model identity + criterion column, and groups them into `{model: {subject: [columns…]}}`. Chi²-Noise (`NoiseGain-RewardRate`) and Chi²-Bound (`Bound-RewardRate` + `_scaledB`) collapse to one model via the drift alias; asymmetric-LR fits are their own model entries.

In [3]:
from .model import compare

df_behavior = compare.prepare_behavior_df()
fits = compare.discover_fits()

Nullifying: 1,271/82,389 trials with calcStimulusTime > 4.8s (of which 73 are valid trials)
Calculate average reward rate (and create SessId column)...
Reduce dataframe size to speed up df operations...
Extend trials so all sessions have the same number of trials...
Discovered 4 model(s), 80 subject×model combos.


## Configuration

Everything tunable lives here. Toggle any row in `ROW_FLAGS`; the header row is always shown, and the α/Q and β/R rows additionally require the model to learn that quantity.

In [4]:
from pathlib import Path
from .model.initvals import MLE_TERMINAL_C

# Figure sizing. Grid size = (n_cols * FIG_COL_WIDTH) x (n_rows * FIG_ROW_HEIGHT).
FIG_COL_WIDTH = 4
FIG_ROW_HEIGHT = 3
DPI = 100

# run_batch writes {model}_{subject}.{IMG_EXT} here.
OUTPUT_DIR = Path("../../results/RLModel/fig_model_cmp")
IMG_EXT = "svg"

# MLE re-evaluation settings applied uniformly to EVERY column so the
# per-column "MLE-Score" (header) and the loss-distribution rows are
# comparable across criteria:
#   MLE_SCORE_TERMINAL_C      : terminal-time no-decision band C for the re-eval.
#   MLE_SCORE_LAPSE_OVERRIDE  : None -> use each fit's own lambda (chisq -> 0);
#                               a float forces the same lambda on every column.
MLE_SCORE_TERMINAL_C = MLE_TERMINAL_C.Default
MLE_SCORE_LAPSE_OVERRIDE = None

# Toggle any diagnostic row on/off.
ROW_FLAGS = {
    "losses_dist": True,     # per-trial -loglik distribution
    "hist_by_loss": True,    # RT hist coloured by loss (red = outliers)
    "rt_corr_incorr": True,  # RT hist (correct / incorrect)
    "rt_direction": True,    # RT hist (left / right)
    "psychometric": True,    # fast/slow psychometric
    "reward_rate": True,     # reward-rate vs RT
    "beta_dist": True,       # reward-rate distribution (R-learning only)
    "alpha_dist": True,      # Q-value distribution (Q-learning only)
    "bias_dist": True,       # starting-point (bias) distribution
}

## Interactive — one subject × model

Pick a **Model** and **Subject**; the grid re-renders below the dropdowns.

In [ ]:
_viewer = compare.interactive_viewer(
    fits, df_behavior, row_flags=ROW_FLAGS,
    fig_col_width=FIG_COL_WIDTH, fig_row_height=FIG_ROW_HEIGHT, dpi=DPI,
    mle_score_terminal_c=MLE_SCORE_TERMINAL_C,
    mle_score_lapse_override=MLE_SCORE_LAPSE_OVERRIDE,
)

Output()

## Batch — every subject × model to disk

Renders and saves one figure per combination to `OUTPUT_DIR`. This runs two forward passes (MLE re-eval + Chi²-style simulation) per column, so it is the slow path — progress prints per figure.

In [6]:
written = compare.run_batch(
    fits, df_behavior, row_flags=ROW_FLAGS,
    out_dir=OUTPUT_DIR, img_ext=IMG_EXT,
    fig_col_width=FIG_COL_WIDTH, fig_row_height=FIG_ROW_HEIGHT, dpi=DPI,
    mle_score_terminal_c=MLE_SCORE_TERMINAL_C,
    mle_score_lapse_override=MLE_SCORE_LAPSE_OVERRIDE,
)

Rendering 80 subject×model grids → ..\..\results\RLModel\fig_model_cmp
Copy df time: 1.35
Copy df time: 1.06
  [1/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_GP4-80.svg
Copy df time: 1.09
Copy df time: 1.25
  [2/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_GP4-24.svg
Copy df time: 1.25
Copy df time: 1.32
  [3/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_Avgat2.svg
Copy df time: 1.14
Copy df time: 1.17
  [4/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_BVGAT1.svg
Copy df time: 1.16
Copy df time: 1.08
  [5/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_GP4-81.svg
Copy df time: 1.10
Copy df time: 1.15
  [6/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_Avgat1.svg
Copy df time: 1.22
Copy df time: 1.06
  [7/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_RDK_WT6.svg
Copy df time: 1.06
Copy df time: 1.05
  [8/80] wrote RewardRate - None_ - Normal(0, 1) - 4.8s_GP4-85.svg
Copy df time: 1.03
Copy df time: 1.07
  [9/80] wrote RewardRate - None_ - Normal(0, 1) -